# Python并发编程（Futures篇）
## 学习目标
- 区分并发与并行的概念
- 理解全局解释器锁（GIL）的影响
- 掌握ThreadPoolExecutor的使用
- 理解Future的原理和常用方法
- 学会选择合适的并发方案

## 一、核心概念
### 1. 并发（Concurrency）与并行（Parallelism）
| 概念 | 含义 | Python实现 | 适用场景 |
|------|------|------------|----------|
| 并发 | 交替执行（同一时刻只允许一个操作） | threading、asyncio | I/O密集型 |
| 并行 | 同时执行（多核同时计算） | multiprocessing | CPU密集型 |

### 2. 全局解释器锁（GIL）
- Python解释器不是线程安全的
- 引入GIL，同一时刻只允许一个线程执行
- I/O操作时释放锁，让其他线程继续执行
- 多线程适用于I/O密集型，多进程适用于CPU密集型

## 二、单线程 vs 多线程性能比较
### 1. 单线程版本（效率低下）

In [ ]:
"""单线程下载（效率低下）"""
import time
import requests

def download_one(url):
    resp = requests.get(url)
    print(f"Read {len(resp.content)} from {url}")

def download_all(sites):
    for site in sites:
        download_one(site)

def main():
    sites = [
        "https://www.example.com",
        "https://www.python.org",
        "https://www.github.com"
    ]
    start = time.perf_counter()
    download_all(sites)
    end = time.perf_counter()
    print(f"总耗时：{end - start:.2f}秒")

# if __name__ == "__main__":
#     main()

### 2. 多线程版本（ThreadPoolExecutor）

In [ ]:
"""多线程下载（效率提升10倍+）"""
import concurrent.futures
import time
import requests

def download_one(url):
    resp = requests.get(url)
    print(f"Read {len(resp.content)} from {url}")

def download_all(sites):
    # 创建线程池，max_workers=5表示最多5个线程
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        executor.map(download_one, sites)

def main():
    sites = [
        "https://www.example.com",
        "https://www.python.org",
        "https://www.github.com"
    ]
    start = time.perf_counter()
    download_all(sites)
    end = time.perf_counter()
    print(f"总耗时：{end - start:.2f}秒")

# if __name__ == "__main__":
#     main()

## 三、Futures的原理和常用函数
### 1. 什么是Futures？
Futures将处于等待状态的操作包裹起来放到队列中，这些操作的状态随时可以查询，结果或异常在操作完成后获取。

### 2. 常用函数
| 函数 | 说明 |
|------|------|
| `executor.submit(func)` | 安排函数执行并返回future实例 |
| `future.done()` | 操作是否完成（非阻塞） |
| `future.result()` | 返回操作结果或异常 |
| `future.add_done_callback(fn)` | 操作完成后执行回调函数 |
| `as_completed(fs)` | 返回完成后的迭代器 |
| `executor.map(func, iterable)` | 对每个元素并发调用函数 |

In [ ]:
"""Futures详细用法示例"""
import concurrent.futures
import time
import requests

def download_one(url):
    resp = requests.get(url)
    return len(resp.content)

def download_all(sites):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        to_do = []
        for site in sites:
            # submit安排函数执行，返回future实例
            future = executor.submit(download_one, site)
            to_do.append(future)
        
        # as_completed：完成后返回迭代器
        for future in concurrent.futures.as_completed(to_do):
            print(f"结果：{future.result()} 字节")

# 使用
# download_all(["https://www.example.com", "https://www.python.org"])

## 四、ProcessPoolExecutor（进程池）
适用于CPU密集型场景。

**注意**：
- 系统自动返回CPU数量作为可调用的进程数
- 对于I/O密集操作，多进程不会提升效率
- 因为CPU数量限制，进程数通常远少于线程数

In [ ]:
"""ProcessPoolExecutor使用示例"""
import concurrent.futures

def cpu_heavy_task(n):
    """CPU密集型任务：计算斐波那契数列"""
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def main():
    # 多线程版本（因为GIL，不适合CPU密集型）
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(executor.map(cpu_heavy_task, [100000, 200000, 300000]))
        print("线程池结果：", results)
    
    # 多进程版本（适合CPU密集型）
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(cpu_heavy_task, [100000, 200000, 300000]))
        print("进程池结果：", results)

# if __name__ == "__main__":
#     main()

## 五、练习与自测
### 练习题1：多线程下载器
**题目**：编写一个多线程下载器，从10个URL下载内容，使用ThreadPoolExecutor，打印每个URL的下载结果和总耗时。

In [ ]:
"""练习题1解答：多线程下载器"""
import concurrent.futures
import time
import requests

def download(url):
    """下载单个URL"""
    try:
        resp = requests.get(url, timeout=5)
        return url, len(resp.content), None
    except Exception as e:
        return url, 0, str(e)

def main():
    urls = [
        "https://www.example.com",
        "https://www.python.org",
        "https://www.github.com",
        "https://www.stackoverflow.com",
        "https://www.wikipedia.org"
    ] * 2  # 共10个URL
    
    start = time.perf_counter()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(download, url) for url in urls]
        for future in concurrent.futures.as_completed(futures):
            url, size, error = future.result()
            if error:
                print(f"❌ {url}: {error}")
            else:
                print(f"✅ {url}: {size} 字节")
    
    elapsed = time.perf_counter() - start
    print(f"\n总耗时：{elapsed:.2f}秒")

# if __name__ == "__main__":
#     main()

### 练习题2：选择正确的并发方案
**题目**：以下场景应该选择哪种并发方案？请说明理由。
1. 爬取1000个网页的内容
2. 计算100万个大数的质因数分解
3. 同时读取10个CSV文件并汇总数据

**参考答案**：
1. I/O密集型 → **协程（asyncio）** 或 **多线程**
2. CPU密集型 → **多进程（multiprocessing）**
3. I/O密集型（文件读写也属于I/O）→ **多线程**

## 六、知识总结
### 重点记忆
1. **并发 vs 并行**
   - 并发：交替执行，I/O密集型
   - 并行：同时执行，CPU密集型

2. **GIL（全局解释器锁）**
   - 同一时刻只允许一个线程执行
   - I/O操作时释放锁
   - 多线程适合I/O，多进程适合CPU

3. **ThreadPoolExecutor vs ProcessPoolExecutor**
   | 特性 | ThreadPoolExecutor | ProcessPoolExecutor |
   |------|-------------------|-------------------|
   | 适用场景 | I/O密集型 | CPU密集型 |
   | 进程/线程数 | 可自定义 | 自动=CPU核数 |
   | 共享变量 | 需要锁 | 不共享 |

4. **Future常用函数**
   - submit / map / done / result / as_completed / add_done_callback

5. **选型建议**
   - I/O密集型：asyncio > threading > multiprocessing
   - CPU密集型：multiprocessing > threading
   - 不要轻易炫技，根据工程需求选择合适方案

## 深入GIL全局解释器锁
### 1. GIL是什么？
GIL（Global Interpreter Lock）是CPython解释器中的一个技术术语，本质上是类似操作系统的Mutex（互斥锁）。

**为什么需要GIL？**
1. CPython使用引用计数管理内存，GIL避免race condition
2. CPython大量使用C语言库，大部分不是线程安全的

### 2. GIL的工作原理
GIL的工作机制可以用以下伪代码理解：

In [ ]:
"""GIL的工作原理：CPU密集任务多线程反而更慢！"""
from threading import Thread
import time

def count_down(n):
    """CPU密集型任务"""
    while n > 0:
        n -= 1

n = 100000000

# 单线程
start = time.time()
count_down(n)
print(f"单线程耗时：{time.time() - start:.2f}s")  # 约5.4s

# 双线程
t1 = Thread(target=count_down, args=[n // 2])
t2 = Thread(target=count_down, args=[n // 2])
start = time.time()
t1.start(); t2.start()
t1.join(); t2.join()
print(f"双线程耗时：{time.time() - start:.2f}s")  # 约9.6s（更慢！）
# 原因：GIL导致线程切换开销，且同一时刻只有一个线程执行

### 3. GIL下的线程安全
**有了GIL不等于不需要考虑线程安全！**

In [ ]:
"""GIL不能保证线程安全"""
import threading

n = 0
def foo():
    global n
    n += 1  # 这一行实际由4条bytecode组成！

threads = []
for _ in range(100):
    t = threading.Thread(target=foo)
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"n = {n}")  # 可能不是100！
# 因为n+=1不是原子操作
# LOAD_GLOBAL → LOAD_CONST → INPLACE_ADD → STORE_GLOBAL
# 这4条bytecode之间可能被打断！

In [ ]:
"""正确做法：使用Lock保证线程安全"""
import threading

n = 0
lock = threading.Lock()

def foo_safe():
    global n
    with lock:  # 加锁，保证原子操作
        n += 1

threads = []
for _ in range(100):
    t = threading.Thread(target=foo_safe)
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"n = {n}")  # 100（正确）

### 4. 如何绕过GIL？
| 方法 | 说明 | 例子 |
|------|------|------|
| 使用别的Python实现 | 绕过CPython | JPython、IronPython |
| 把关键代码放到其他语言 | C/C++实现 | NumPy、自定义C扩展 |

**实际应用**：
- NumPy的矩阵运算通过C实现，不受GIL影响
- 深度学习框架的关键部分用C++实现
- 大部分情况下不需要过多考虑GIL